# Chapter 2: Contact Manifolds

**Source span:** printed pp. 51-92; physical PDF pp. 69-110.

## Chapter Question

Once contact forms are available, what moves let us compare them, flow them, and normalize them? Chapter 2 builds the differential-topological toolkit: examples, Gray stability, the Moser trick, contact Hamiltonians, Darboux's theorem, neighborhood theorems, and isotopy extension. This notebook follows the computational spine of that toolkit. A vector field is selected so that changing contact data remains controlled.

The local model is still `alpha = dz + x dy`, but the emphasis changes. Chapter 1 asked whether `alpha` is contact. Here we ask which vector fields preserve or deliberately deform contact data. The central example is a contact Hamiltonian whose projected motion is easy to read: rotation in the `xy` plane paired with a vertical correction that makes the contact equation true.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/An-Introduction-to-Contact-Topology/chapter-02-contact-manifolds/02-contact-manifolds.ipynb",
  "course_dir": "An-Introduction-to-Contact-Topology",
  "course_title": "An Introduction to Contact Topology",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/An-Introduction-to-Contact-Topology/chapter-02-contact-manifolds/02-contact-manifolds.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "An-Introduction-to-Contact-Topology/chapter-02-contact-manifolds/02-contact-manifolds.ipynb",
  "notebook_title": "Chapter 2: Contact Manifolds",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation Guide

Gray stability says that a smooth family of contact structures on a closed manifold is locally produced by an isotopy. Computationally, that means solving for a time-dependent vector field. The Moser trick is the design pattern: differentiate the condition to be preserved, then solve a linear equation for a vector field that cancels the unwanted change. A contact Hamiltonian is a function `H` encoding a contact vector field, but the formula differs from symplectic Hamiltonian mechanics because the Reeb direction and the condition `alpha(X_H)=H` are part of the data.

Darboux's theorem says the local model has no local invariants. Neighborhood and isotopy extension theorems turn submanifold data into ambient contact data. The code below keeps the sign convention visible because later chapters reuse these equations in knots, convex surfaces, surgery, and fillings.


In [ ]:
from pathlib import Path
import sys
BOOK_ROOT = Path.cwd()
while not (BOOK_ROOT / "AGENTS.md").exists():
    if BOOK_ROOT.parent == BOOK_ROOT:
        raise RuntimeError("Could not locate course root")
    BOOK_ROOT = BOOK_ROOT.parent
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib
UNIT="chapter-02"; ARTIFACTS=[]; CHECKS=[]


## Visual 1: A Contact Hamiltonian Flow in the Standard Model

Take `H=(x^2+y^2)/2`. For `alpha=dz+x dy`, the contact Hamiltonian equations give `X_H=-y partial_x + x partial_y + (y^2-x^2)/2 partial_z`. Its `xy` projection rotates around the origin, while the `z` component changes sign across the diagonals. The figure shows the projected flow and colors the background by the vertical correction.

This representation separates familiar planar rotation from the extra contact bookkeeping. The inspection target is the mismatch between pure planar motion and the contact lift: the vector field must satisfy both `alpha(X_H)=H` and the contraction equation with `d alpha`.


In [ ]:
grid=np.linspace(-2,2,25); X,Y=np.meshgrid(grid,grid)
U,V=-Y,X; ZCORR=0.5*(Y**2-X**2)
fig,ax=plt.subplots(figsize=(7,6))
contour=ax.contourf(X,Y,ZCORR,levels=16,cmap="coolwarm",alpha=0.72)
ax.streamplot(X,Y,U,V,color="black",density=1.05,linewidth=0.75,arrowsize=0.9)
ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Projected contact Hamiltonian flow; color is z-correction")
fig.colorbar(contour,ax=ax,label="(y^2 - x^2)/2")
flow_path=save_matplotlib(fig,UNIT,"figures","contact-hamiltonian-flow.png")
plt.close(fig); ARTIFACTS.append(flow_path); display_artifact(flow_path,width=720)


## Symbolic Check: The Contact Hamiltonian Equations

With `d alpha = dx wedge dy`, a vector field `a partial_x + b partial_y + c partial_z` has contraction `a dy - b dx`. For `H=(x^2+y^2)/2` and no `z` dependence, the right side is `-dH`. The proposed vector field has `a=-y`, `b=x`, and `c=(y^2-x^2)/2`. The code checks both equations and stores the residuals. Zero residuals are the computational analogue of having chosen the Moser/contact vector field correctly.


In [ ]:
x,y,z=sp.symbols("x y z")
H=(x**2+y**2)/2
a,b,c=-y,x,(y**2-x**2)/2
alpha_on_X=sp.simplify(c+x*b)
residuals={
 "alpha_X_minus_H":str(sp.simplify(alpha_on_X-H)),
 "iX_dalpha_dx_residual":str(sp.simplify(-b + sp.diff(H,x))),
 "iX_dalpha_dy_residual":str(sp.simplify(a + sp.diff(H,y))),
 "vector_field":"-y partial_x + x partial_y + (y^2-x^2)/2 partial_z"}
json_path=save_json(residuals,UNIT,"checks","contact-hamiltonian-residuals.json")
CHECKS.append(json_path); residuals


## Applied Lab

The table compares three Hamiltonians. A constant Hamiltonian is pure Reeb motion. A linear Hamiltonian tilts the flow across contact planes. The quadratic Hamiltonian above produces the rotational picture. The exercise is not to memorize formulas, but to notice the design constraint: the horizontal part is chosen from the derivatives of `H`, and the vertical part is then forced by `alpha(X_H)=H`.


For **02 Contact Manifolds**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
hamiltonians=[sp.Integer(1),x,(x**2+y**2)/2]
rows=[]
for item in hamiltonians:
    a_item=-sp.diff(item,y); b_item=sp.diff(item,x); c_item=sp.simplify(item-x*b_item)
    rows.append({"H":str(item),"a":str(a_item),"b":str(b_item),"c":str(c_item)})
lab_json=save_json(rows,UNIT,"checks","hamiltonian-comparison-table.json")
CHECKS.append(lab_json); rows


## Takeaways

The chapter's stability and normal-form theorems are powered by a repeated maneuver: translate a geometric preservation problem into an equation for a vector field. Contact Hamiltonians make that maneuver explicit in the standard model. Darboux's theorem tells us that every contact manifold is locally this model, while Gray stability and isotopy extension explain when contact data can be moved without changing its essential global type. The same equations later control Legendrian neighborhoods, convex-surface isotopies, surgery attachments, and symplectic collars.


In **02 Contact Manifolds**, the important habit is to connect the source terminology to a visible object, then read the diagnostic as a small proof obligation.

## Source-Specific Inspection Notes

This enrichment note is specific to **02 Contact Manifolds**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **02 Contact Manifolds**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


## Additional Source span Inspection Contract

Source span review for **02 Contact Manifolds**: inspect the local chapter map, then read the notebook visual as a compact model of that span. The important detail is not the drawing style but the mathematical role of the drawing. Ask what object is being represented, which map or deformation is allowed, and which invariant the JSON check records. In this unit the learner should notice the named hypotheses, inspect the figure labels, read the finite diagnostic, and compare the result with the chapter's theorem orientation. If the diagnostic is stable, explain which assumption protects it. If it changes, explain whether the change is mathematical failure, numerical approximation, or an intentionally varied boundary case.

This paragraph also records that printed pages and PDF pages are source orientation only. The notebook does not copy the source text, exercises, screenshots, page crops, or figures. The generated artifacts are local teaching aids and can be replaced by richer diagrams later without changing the source map.


In [ ]:
for path in ARTIFACTS:
    assert_artifact(path,min_bytes=1500)
for path in CHECKS:
    assert_artifact(path,min_bytes=80)
assert all(value=="0" for key,value in residuals.items() if key.endswith("residual") or key=="alpha_X_minus_H")
print("Chapter 2 Hamiltonian flow checks passed.")
